In [1]:
!pip install pandas scikit-learn matplotlib seaborn numpy

In [3]:
"""
Heart Disease Prediction — Logistic Regression
================================================
Dataset : Heart Disease UCI (heart_disease_uci.csv)
Model   : Logistic Regression (L2, balanced class weights)
Metrics : Accuracy, ROC-AUC, Confusion Matrix, Classification Report
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
)
from sklearn.impute import SimpleImputer


# ── 1. Load ──────────────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Loaded  : {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Columns : {list(df.columns)}\n")
    return df


# ── 2. Clean & encode ─────────────────────────────────────────────────────────
def preprocess(df: pd.DataFrame):
    df = df.drop(columns=["id", "dataset"], errors="ignore")

    # Binary target: 0 = healthy, 1 = heart disease
    df["target"] = (df["num"] > 0).astype(int)
    df = df.drop(columns=["num"])

    # Boolean columns → int
    for col in ["fbs", "exang"]:
        if col in df.columns:
            df[col] = df[col].map({"TRUE": 1, "FALSE": 0, True: 1, False: 0})

    # Label-encode remaining object columns
    le = LabelEncoder()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("Unknown")
        df[col] = le.fit_transform(df[col].astype(str))

    df = df.apply(pd.to_numeric, errors="coerce")

    # Impute missing values with column medians
    missing = df.isnull().sum()
    if missing.any():
        print("Missing values (before imputation):")
        print(missing[missing > 0].to_string(), "\n")

    imp = SimpleImputer(strategy="median")
    df_clean = pd.DataFrame(imp.fit_transform(df), columns=df.columns)

    print(f"Target distribution:\n{df_clean['target'].value_counts().to_string()}\n")
    return df_clean


# ── 3. Train / evaluate ───────────────────────────────────────────────────────
def train_and_evaluate(df: pd.DataFrame):
    X = df.drop("target", axis=1)
    y = df["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
        C=1.0,
    )
    model.fit(X_train_s, y_train)

    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    cm  = confusion_matrix(y_test, y_pred)
    cr  = classification_report(y_test, y_pred, target_names=["No Disease", "Disease"])

    print("=" * 55)
    print(f"  Accuracy  : {acc:.4f}  ({acc:.1%})")
    print(f"  ROC-AUC   : {roc:.4f}")
    print("=" * 55)
    print("\nClassification Report:\n")
    print(cr)
    print("Confusion Matrix:")
    print(cm, "\n")

    # Feature importance (absolute standardized coefficients)
    coef_df = (
        pd.Series(model.coef_[0], index=X.columns)
        .abs()
        .sort_values(ascending=False)
        .rename("importance")
    )
    print("Feature Importance (|coefficient|):")
    print(coef_df.to_string(), "\n")

    return model, scaler, X, y_test, y_pred, y_prob, cm, coef_df


# ── 4. Entry point ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    DATA_PATH = "heart_disease_uci.csv"   # adjust if needed

    df_raw   = load_data(DATA_PATH)
    df_clean = preprocess(df_raw)
    model, scaler, X, y_test, y_pred, y_prob, cm, coef_df = train_and_evaluate(df_clean)

Loaded  : 920 rows × 16 columns
Columns : ['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

Missing values (before imputation):
trestbps     59
chol         30
fbs          90
thalch       55
exang        55
oldpeak      62
ca          611 

Target distribution:
target
1.0    509
0.0    411

  Accuracy  : 0.8098  (81.0%)
  ROC-AUC   : 0.8940

Classification Report:

              precision    recall  f1-score   support

  No Disease       0.78      0.80      0.79        82
     Disease       0.84      0.81      0.83       102

    accuracy                           0.81       184
   macro avg       0.81      0.81      0.81       184
weighted avg       0.81      0.81      0.81       184

Confusion Matrix:
[[66 16]
 [19 83]] 

Feature Importance (|coefficient|):
oldpeak     0.592389
exang       0.574106
sex         0.544844
ca          0.539572
cp          0.499849
chol        0.472855
thalch      0.39

In [15]:
"""
Heart Disease — Exploratory Data Analysis (Tab 1 & 2)
======================================================
Generates:
  tab1_overview.png    — target dist, age hist, sex, missing vals,
                         cholesterol boxplot, age-vs-heartrate scatter
  tab2_correlation.png — correlation heatmap + target correlations
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# ── Palette ───────────────────────────────────────────────────────────────────
BG   = "#0d1117"
CARD = "#161b22"
ACC1 = "#e84393"   # pink  — disease
ACC2 = "#00d2ff"   # cyan  — healthy
ACC3 = "#f9a825"   # amber
ACC4 = "#69ff47"   # lime
TEXT = "#e6edf3"
MUTED = "#8b949e"
GRID = "#21262d"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": CARD,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": TEXT, "grid.color": GRID,
    "grid.linewidth": 0.5,
})


def load_and_prepare(path: str):
    df = pd.read_csv(path)
    df_orig = df.copy()  # keep for categorical charts

    df = df.drop(columns=["id", "dataset"], errors="ignore")
    df["target"] = (df["num"] > 0).astype(int)
    df = df.drop(columns=["num"])

    for col in ["fbs", "exang"]:
        if col in df.columns:
            df[col] = df[col].map({"TRUE": 1, "FALSE": 0, True: 1, False: 0})

    le = LabelEncoder()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("Unknown")
        df[col] = le.fit_transform(df[col].astype(str))

    df = df.apply(pd.to_numeric, errors="coerce")
    imp = SimpleImputer(strategy="median")
    df_imp = pd.DataFrame(imp.fit_transform(df), columns=df.columns)
    return df_imp, df_orig


# ── Tab 1 — Overview ──────────────────────────────────────────────────────────
def plot_overview(df_imp: pd.DataFrame, df_orig: pd.DataFrame, out: str = "tab1_overview.png"):
    y = df_imp["target"]
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.patch.set_facecolor(BG)
    fig.suptitle("TAB 1 — DATASET OVERVIEW & EDA", fontsize=20,
                 fontweight="bold", color=TEXT, y=1.01)

    # 1a Target distribution
    ax = axes[0, 0]
    counts = y.value_counts()
    bars = ax.bar(["No Disease (0)", "Heart Disease (1)"],
                  [counts[0], counts[1]], color=[ACC2, ACC1],
                  width=0.5, edgecolor=BG, linewidth=2)
    for bar, cnt in zip(bars, [counts[0], counts[1]]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
                f"{cnt}\n({cnt/len(y)*100:.1f}%)", ha="center", va="bottom",
                color=TEXT, fontsize=12, fontweight="bold")
    ax.set_title("Target Distribution", color=TEXT, fontweight="bold")
    ax.set_ylabel("Count"); ax.grid(axis="y", alpha=0.4)
    ax.set_ylim(0, max(counts) * 1.18)

    # 1b Age histogram
    ax = axes[0, 1]
    ax.hist(df_imp[y == 0]["age"], bins=20, color=ACC2, alpha=0.75,
            label="No Disease", edgecolor=BG)
    ax.hist(df_imp[y == 1]["age"], bins=20, color=ACC1, alpha=0.75,
            label="Heart Disease", edgecolor=BG)
    ax.set_title("Age Distribution by Class", color=TEXT, fontweight="bold")
    ax.set_xlabel("Age"); ax.set_ylabel("Count")
    ax.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    ax.grid(axis="y", alpha=0.4)

    # 1c Sex breakdown
    ax = axes[0, 2]
    sex_counts = df_imp.groupby(["sex", "target"]).size().unstack(fill_value=0)
    x_pos = np.arange(len(sex_counts)); w = 0.35
    ax.bar(x_pos - w/2, sex_counts[0], w, color=ACC2, label="No Disease", edgecolor=BG)
    ax.bar(x_pos + w/2, sex_counts[1], w, color=ACC1, label="Heart Disease", edgecolor=BG)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"Sex {int(i)}" for i in sex_counts.index])
    ax.set_title("Disease by Sex", color=TEXT, fontweight="bold")
    ax.set_ylabel("Count")
    ax.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    ax.grid(axis="y", alpha=0.4)

    # 1d Missing values
    ax = axes[1, 0]
    miss = df_orig.isnull().sum().sort_values(ascending=False)
    miss = miss[miss > 0]
    if len(miss):
        colors_m = [ACC1 if v > 100 else ACC3 if v > 30 else ACC4 for v in miss]
        ax.barh(miss.index, miss.values, color=colors_m, edgecolor=BG)
        for i, v in enumerate(miss.values):
            ax.text(v + 2, i, f"{v} ({v/len(df_orig)*100:.1f}%)",
                    va="center", color=TEXT, fontsize=9)
    ax.set_title("Missing Values per Feature", color=TEXT, fontweight="bold")
    ax.set_xlabel("Count"); ax.grid(axis="x", alpha=0.4)
    if len(miss):
        ax.set_xlim(0, miss.max() * 1.25)

    # 1e Cholesterol boxplot
    ax = axes[1, 1]
    bp = ax.boxplot([df_imp[y == 0]["chol"], df_imp[y == 1]["chol"]],
                    patch_artist=True,
                    medianprops=dict(color=BG, linewidth=2),
                    whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
                    flierprops=dict(marker="o", markerfacecolor=ACC3,
                                   markersize=4, alpha=0.5))
    bp["boxes"][0].set_facecolor(ACC2); bp["boxes"][1].set_facecolor(ACC1)
    ax.set_xticklabels(["No Disease", "Heart Disease"])
    ax.set_title("Cholesterol vs Disease Status", color=TEXT, fontweight="bold")
    ax.set_ylabel("Cholesterol (mg/dl)"); ax.grid(axis="y", alpha=0.4)

    # 1f Age vs max heart rate
    ax = axes[1, 2]
    ax.scatter(df_imp[y == 0]["age"], df_imp[y == 0]["thalch"],
               c=ACC2, alpha=0.5, s=25, label="No Disease")
    ax.scatter(df_imp[y == 1]["age"], df_imp[y == 1]["thalch"],
               c=ACC1, alpha=0.5, s=25, label="Heart Disease")
    ax.set_title("Age vs Max Heart Rate", color=TEXT, fontweight="bold")
    ax.set_xlabel("Age"); ax.set_ylabel("Max Heart Rate (thalch)")
    ax.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Saved → {out}")


# ── Tab 2 — Correlation ───────────────────────────────────────────────────────
def plot_correlation(df_imp: pd.DataFrame, out: str = "tab2_correlation.png"):
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.patch.set_facecolor(BG)
    fig.suptitle("TAB 2 — CORRELATION & FEATURE ANALYSIS", fontsize=20,
                 fontweight="bold", color=TEXT)

    # 2a Heatmap
    ax = axes[0]
    corr = df_imp.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    cmap = sns.diverging_palette(220, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, ax=ax, cmap=cmap, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 8, "color": TEXT},
                linewidths=0.5, linecolor=BG, cbar_kws={"shrink": 0.8})
    ax.set_facecolor(CARD)
    ax.set_title("Feature Correlation Matrix", color=TEXT, fontweight="bold", pad=12)
    ax.tick_params(labelsize=8, colors=MUTED)

    # 2b Target correlations
    ax = axes[1]
    target_corr = df_imp.corr()["target"].drop("target").sort_values()
    colors_corr = [ACC1 if v > 0 else ACC2 for v in target_corr]
    bars = ax.barh(target_corr.index, target_corr.values,
                   color=colors_corr, edgecolor=BG, height=0.6)
    for bar, val in zip(bars, target_corr.values):
        ax.text(val + (0.005 if val >= 0 else -0.005),
                bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center",
                ha="left" if val >= 0 else "right",
                color=TEXT, fontsize=9)
    ax.axvline(0, color=MUTED, linewidth=1.5, linestyle="--")
    ax.set_title("Feature Correlation with Target", color=TEXT, fontweight="bold")
    ax.set_xlabel("Pearson Correlation"); ax.grid(axis="x", alpha=0.3)
    p1 = mpatches.Patch(color=ACC1, label="Positive (risk ↑)")
    p2 = mpatches.Patch(color=ACC2, label="Negative (protective)")
    ax.legend(handles=[p1, p2], facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)

    plt.tight_layout()
    plt.show()  # Add this
    #plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    #plt.close()
    print(f"Saved → {out}")


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    DATA_PATH = "heart_disease_uci.csv"   # adjust if needed

    df_imp, df_orig = load_and_prepare(DATA_PATH)
    plot_overview(df_imp, df_orig)
    plot_correlation(df_imp)

Saved → tab1_overview.png
Saved → tab2_correlation.png


In [5]:
"""
Heart Disease — Feature Importance (Tab 4)
==========================================
Generates tab4_features.png:
  • Signed logistic regression coefficients (standardized)
  • Absolute importance bar chart (ranked)
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

BG    = "#0d1117"
CARD  = "#161b22"
ACC1  = "#e84393"
ACC2  = "#00d2ff"
TEXT  = "#e6edf3"
MUTED = "#8b949e"
GRID  = "#21262d"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": CARD,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": TEXT, "grid.color": GRID, "grid.linewidth": 0.5,
})

BAR_PALETTE = [
    ACC1, "#f9a825", ACC2, "#b388ff", "#ff8a65", "#80cbc4",
    "#f48fb1", "#e6ee9c", "#80deea", "#ffcc80", "#ce93d8", "#a5d6a7",
]


def prepare_and_train(path: str):
    df = pd.read_csv(path).drop(columns=["id", "dataset"], errors="ignore")
    df["target"] = (df["num"] > 0).astype(int)
    df = df.drop(columns=["num"])
    for col in ["fbs", "exang"]:
        if col in df.columns:
            df[col] = df[col].map({"TRUE": 1, "FALSE": 0, True: 1, False: 0})
    le = LabelEncoder()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("Unknown")
        df[col] = le.fit_transform(df[col].astype(str))
    df = df.apply(pd.to_numeric, errors="coerce")
    imp = SimpleImputer(strategy="median")
    df = pd.DataFrame(imp.fit_transform(df), columns=df.columns)

    X, y = df.drop("target", axis=1), df["target"]
    X_tr, X_te, y_tr, _ = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)

    model = LogisticRegression(max_iter=1000, random_state=42,
                               class_weight="balanced", C=1.0)
    model.fit(X_tr_s, y_tr)

    coefs = pd.Series(model.coef_[0], index=X.columns).sort_values()
    abs_coef = coefs.abs().sort_values(ascending=False)
    return coefs, abs_coef


def plot_feature_importance(coefs: pd.Series, abs_coef: pd.Series,
                            out: str = "tab4_features.png"):
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.patch.set_facecolor(BG)
    fig.suptitle("TAB 4 — FEATURE IMPORTANCE ANALYSIS",
                 fontsize=20, fontweight="bold", color=TEXT)

    # 4a Signed coefficients
    ax = axes[0]
    colors_coef = [ACC1 if v > 0 else ACC2 for v in coefs]
    ax.barh(coefs.index, coefs.values, color=colors_coef, edgecolor=BG, height=0.6)
    for i, (idx, val) in enumerate(coefs.items()):
        ax.text(val + (0.02 if val >= 0 else -0.02), i,
                f"{val:+.3f}", va="center",
                ha="left" if val >= 0 else "right", color=TEXT, fontsize=9)
    ax.axvline(0, color=MUTED, linewidth=1.5, linestyle="--")
    ax.set_title("Logistic Regression Coefficients\n(after standardization)",
                 color=TEXT, fontweight="bold")
    ax.set_xlabel("Coefficient Value"); ax.grid(axis="x", alpha=0.3)
    p1 = mpatches.Patch(color=ACC1, label="Increases risk")
    p2 = mpatches.Patch(color=ACC2, label="Decreases risk")
    ax.legend(handles=[p1, p2], facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)

    # 4b Absolute importance
    ax = axes[1]
    colors_abs = BAR_PALETTE[:len(abs_coef)]
    bars = ax.bar(abs_coef.index, abs_coef.values,
                  color=colors_abs, edgecolor=BG, width=0.6)
    for bar, val in zip(bars, abs_coef.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom",
                color=TEXT, fontsize=8.5, fontweight="bold")
    ax.set_title("Feature Importance (|Coefficient|)", color=TEXT, fontweight="bold")
    ax.set_ylabel("Absolute Coefficient"); ax.set_xlabel("Feature")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    ax.grid(axis="y", alpha=0.3); ax.set_ylim(0, abs_coef.max() * 1.15)

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Saved → {out}")
    print("\nTop 5 features:")
    for feat, val in abs_coef.head(5).items():
        direction = "risk ↑" if coefs[feat] > 0 else "protective ↓"
        print(f"  {feat:12s}  |coeff| = {val:.4f}  ({direction})")


if __name__ == "__main__":
    DATA_PATH = "heart_disease_uci.csv"
    coefs, abs_coef = prepare_and_train(DATA_PATH)
    plot_feature_importance(coefs, abs_coef)

Saved → tab4_features.png

Top 5 features:
  oldpeak       |coeff| = 0.5924  (risk ↑)
  exang         |coeff| = 0.5741  (risk ↑)
  sex           |coeff| = 0.5448  (risk ↑)
  ca            |coeff| = 0.5396  (risk ↑)
  cp            |coeff| = 0.4998  (protective ↓)


In [7]:
"""
Heart Disease — Clinical Insights (Tab 5)
==========================================
Generates tab5_insights.png:
  • Chest pain type vs disease
  • Thalassemia type vs disease
  • Predicted probability distribution + decision boundaries
  • Age-group risk prevalence
  • Precision / Recall / F1 per class
  • Model summary card
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, accuracy_score, roc_curve, classification_report,
)
from sklearn.impute import SimpleImputer

BG    = "#0d1117"
CARD  = "#161b22"
ACC1  = "#e84393"
ACC2  = "#00d2ff"
ACC3  = "#f9a825"
ACC4  = "#69ff47"
TEXT  = "#e6edf3"
MUTED = "#8b949e"
GRID  = "#21262d"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": CARD,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": TEXT, "grid.color": GRID, "grid.linewidth": 0.5,
})


def prepare_and_train(path: str):
    df_orig = pd.read_csv(path)
    df_orig["target"] = (df_orig["num"] > 0).astype(int)

    df = df_orig.copy().drop(columns=["id", "dataset"], errors="ignore")
    df = df.drop(columns=["num"])
    for col in ["fbs", "exang"]:
        if col in df.columns:
            df[col] = df[col].map({"TRUE": 1, "FALSE": 0, True: 1, False: 0})
    le = LabelEncoder()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("Unknown")
        df[col] = le.fit_transform(df[col].astype(str))
    df = df.apply(pd.to_numeric, errors="coerce")
    imp = SimpleImputer(strategy="median")
    df_imp = pd.DataFrame(imp.fit_transform(df), columns=df.columns)

    X, y = df_imp.drop("target", axis=1), df_imp["target"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

    model = LogisticRegression(max_iter=1000, random_state=42,
                               class_weight="balanced", C=1.0)
    model.fit(X_tr_s, y_tr)

    y_pred = model.predict(X_te_s)
    y_prob = model.predict_proba(X_te_s)[:, 1]

    coefs = pd.Series(model.coef_[0], index=X.columns)
    abs_coef = coefs.abs().sort_values(ascending=False)

    return df_imp, df_orig, y_te, y_pred, y_prob, abs_coef, X_tr, X_te, model


def plot_insights(df_imp, df_orig, y_te, y_pred, y_prob, abs_coef,
                  X_tr, X_te, model, out: str = "tab5_insights.png"):
    acc = accuracy_score(y_te, y_pred)
    roc = roc_auc_score(y_te, y_prob)
    cr  = classification_report(y_te, y_pred, output_dict=True)
    fpr, tpr, thresh = roc_curve(y_te, y_prob)
    best_ix = np.argmax(tpr - fpr)
    ck = sorted([k for k in cr if k not in ("accuracy", "macro avg", "weighted avg")],
                key=float)

    fig, axes = plt.subplots(2, 3, figsize=(19, 11))
    fig.patch.set_facecolor(BG)
    fig.suptitle("TAB 5 — CLINICAL DEEP DIVE & INSIGHTS",
                 fontsize=20, fontweight="bold", color=TEXT, y=1.01)

    # 5a Chest pain type
    ax = axes[0, 0]
    cp_counts = df_orig.groupby(["cp", "target"]).size().unstack(fill_value=0)
    cp_counts.plot(kind="bar", ax=ax, color=[ACC2, ACC1], edgecolor=BG, width=0.65)
    ax.set_title("Chest Pain Type vs Disease", color=TEXT, fontweight="bold")
    ax.set_xlabel("Chest Pain Type"); ax.set_ylabel("Count")
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
    ax.legend(["No Disease", "Heart Disease"],
              facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    ax.grid(axis="y", alpha=0.4)

    # 5b Thalassemia type
    ax = axes[0, 1]
    thal_counts = df_orig.groupby(["thal", "target"]).size().unstack(fill_value=0)
    thal_counts.plot(kind="bar", ax=ax, color=[ACC2, ACC1], edgecolor=BG, width=0.65)
    ax.set_title("Thalassemia Type vs Disease", color=TEXT, fontweight="bold")
    ax.set_xlabel("Thal"); ax.set_ylabel("Count")
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
    ax.legend(["No Disease", "Heart Disease"],
              facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    ax.grid(axis="y", alpha=0.4)

    # 5c Predicted probability distribution
    ax = axes[0, 2]
    ax.hist(y_prob[y_te == 0], bins=25, color=ACC2, alpha=0.75,
            label="No Disease (True)", edgecolor=BG)
    ax.hist(y_prob[y_te == 1], bins=25, color=ACC1, alpha=0.75,
            label="Heart Disease (True)", edgecolor=BG)
    ax.axvline(0.5, color=ACC3, linestyle="--", linewidth=2, label="Boundary (0.5)")
    ax.axvline(thresh[best_ix], color=ACC4, linestyle=":", linewidth=2,
               label=f"Optimal ({thresh[best_ix]:.2f})")
    ax.set_title("Predicted Probability Distribution", color=TEXT, fontweight="bold")
    ax.set_xlabel("Predicted Probability of Disease"); ax.set_ylabel("Count")
    ax.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT, fontsize=8)
    ax.grid(alpha=0.3)

    # 5d Age-group risk
    ax = axes[1, 0]
    df_imp["age_bin"] = pd.cut(df_imp["age"], bins=[25, 35, 45, 55, 65, 80],
                                labels=["26-35", "36-45", "46-55", "56-65", "66-80"])
    risk_by_age = df_imp.groupby("age_bin", observed=True)["target"].mean() * 100
    colors_age = [ACC4 if v < 40 else ACC3 if v < 60 else ACC1
                  for v in risk_by_age.values]
    bars_a = ax.bar(risk_by_age.index, risk_by_age.values,
                    color=colors_age, edgecolor=BG, width=0.6)
    for bar, val in zip(bars_a, risk_by_age.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}%", ha="center", va="bottom",
                color=TEXT, fontsize=11, fontweight="bold")
    ax.set_title("Heart Disease Risk by Age Group", color=TEXT, fontweight="bold")
    ax.set_xlabel("Age Group"); ax.set_ylabel("Disease Prevalence (%)")
    ax.set_ylim(0, risk_by_age.max() * 1.2); ax.grid(axis="y", alpha=0.4)

    # 5e Precision / Recall / F1 per class
    ax = axes[1, 1]
    class_names = ["No Disease", "Disease"]
    precision_vals = [cr[k]["precision"] for k in ck]
    recall_vals    = [cr[k]["recall"]    for k in ck]
    f1_vals        = [cr[k]["f1-score"]  for k in ck]
    x_pos = np.arange(len(class_names)); w = 0.25
    ax.bar(x_pos - w, precision_vals, w, color=ACC2, label="Precision", edgecolor=BG)
    ax.bar(x_pos,     recall_vals,    w, color=ACC1, label="Recall",    edgecolor=BG)
    ax.bar(x_pos + w, f1_vals,        w, color=ACC4, label="F1-Score",  edgecolor=BG)
    ax.set_xticks(x_pos); ax.set_xticklabels(class_names)
    ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
    ax.set_title("Precision / Recall / F1 per Class", color=TEXT, fontweight="bold")
    ax.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    ax.grid(axis="y", alpha=0.4)

    # 5f Model summary card
    ax = axes[1, 2]
    ax.set_facecolor("#0d1117"); ax.axis("off")
    summary = [
        ("MODEL",          "Logistic Regression",             ACC2),
        ("ACCURACY",       f"{acc:.1%}",                      ACC4),
        ("ROC-AUC",        f"{roc:.4f}",                      ACC4),
        ("TRAIN SIZE",     f"{len(X_tr)} samples",            TEXT),
        ("TEST SIZE",      f"{len(X_te)} samples",            TEXT),
        ("FEATURES",       f"{X_tr.shape[1]}",                TEXT),
        ("REGULARIZATION", "L2  (C=1.0)",                     MUTED),
        ("CLASS WEIGHT",   "Balanced",                         MUTED),
        ("",               "",                                 ""),
        ("TOP RISK",       abs_coef.index[0].upper(),         ACC1),
        ("2nd RISK",       abs_coef.index[1].upper(),         ACC3),
        ("3rd RISK",       abs_coef.index[2].upper(),         ACC3),
    ]
    y_pos = 0.97
    for label, val, color in summary:
        if not label:
            y_pos -= 0.04; continue
        ax.text(0.02, y_pos, f"{label}:", transform=ax.transAxes,
                color=MUTED, fontsize=10, va="top")
        ax.text(0.98, y_pos, val, transform=ax.transAxes,
                color=color, fontsize=10, va="top", ha="right", fontweight="bold")
        y_pos -= 0.075
    ax.set_title("Model Summary", color=TEXT, fontweight="bold")
    rect = mpatches.FancyBboxPatch(
        (0.01, 0.01), 0.98, 0.98, boxstyle="round,pad=0.02",
        linewidth=2, edgecolor=ACC1, facecolor="none", transform=ax.transAxes)
    ax.add_patch(rect)

    plt.tight_layout()
    plt.show()
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Saved → {out}")


if __name__ == "__main__":
    DATA_PATH = "heart_disease_uci.csv"
    args = prepare_and_train(DATA_PATH)
    plot_insights(*args)

Saved → tab5_insights.png


In [14]:
def plot_insights(df_imp, df_orig, y_te, y_pred, y_prob, abs_coef,
                  X_tr, X_te, model, out: str = "tab5_insights.png"):
    # ... (your existing plotting code) ...
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()  # Display in Colab
    plt.close()
    print(f"Saved → {out}")

# Run the code
if __name__ == "__main__":
    DATA_PATH = "heart_disease_uci.csv"
    args = prepare_and_train(DATA_PATH)
    plot_insights(*args)



Saved → tab5_insights.png
